In [ ]:
!pip install yfinance pandas requests beautifulsoup4 xlrd


In [4]:
import pandas as pd
import yfinance as yf

# Extract ticker and date range from your URL

ticker_symbol = "KC=F"
start_date = pd.to_datetime(946875600, unit='s').strftime('%Y-%m-%d')
end_date = pd.to_datetime(1789053428, unit='s').strftime('%Y-%m-%d')

print(f"Fetching {ticker_symbol} from {start_date} to {end_date}...")

# Download data
df = yf.download(ticker_symbol, start=start_date, end=end_date)

# Drop secondary level of multi-index columns if present (e.g., Ticker symbol)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# Reset index so 'Date' becomes a column
df.reset_index(inplace=True)

# Save directly to CSV
df.to_csv("data/yahoo/arabica_coffee_futures_history.csv", index=False)

print("Data successfully saved to data/yahoo/arabica_coffee_futures_history.csv")
print(df.head())

[*********************100%***********************]  1 of 1 completed

Fetching KC=F from 2000-01-03 to 2026-09-10...
Data successfully saved to data/yahoo/arabica_coffee_futures_history.csv
Price       Date       Close        High         Low    Open  Volume
0     2000-01-03  116.500000  124.000000  116.099998  124.00    6640
1     2000-01-04  116.250000  120.500000  115.750000  116.50    5492
2     2000-01-05  118.599998  121.000000  115.000000  115.00    6165
3     2000-01-06  116.849998  121.400002  116.500000  119.00    5094
4     2000-01-07  114.150002  117.750000  113.800003  117.75    6855


## CFTC Commitments of Traders compressed datasets

Downloads selected CFTC Historical Compressed COT report ZIPs into dedicated folders under `data/cftc_cot/`.

Default downloads:
- `legacy_futures_only`: primary dataset, yearly Excel ZIPs from 2000 to current year.
- `disaggregated_futures_only`: secondary dataset, CFTC historical bundle for early coverage plus yearly Excel ZIPs from 2017 to current year.

Optional datasets are included in the config but disabled by default. Traders in Financial Futures is intentionally excluded because it is for financial futures, not Coffee C.


In [6]:
from pathlib import Path
from urllib.parse import urljoin, urlparse
import re
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup

CFTC_HISTORICAL_COMPRESSED_URL = "https://www.cftc.gov/MarketReports/CommitmentsofTraders/HistoricalCompressed/index.htm"
CFTC_DATA_DIR = Path("data") / "cftc_cot"
CURRENT_YEAR = pd.Timestamp.today().year

# Turn these on only when you want the optional/supplemental families too.
DOWNLOAD_LEGACY_FUTURES_AND_OPTIONS = False
DOWNLOAD_DISAGGREGATED_FUTURES_AND_OPTIONS = False
DOWNLOAD_COMMODITY_INDEX_TRADER_SUPPLEMENT = False
DOWNLOAD_TEXT_FILES_TOO = False

CFTC_FILE_FORMATS = ["xls"] + (["txt"] if DOWNLOAD_TEXT_FILES_TOO else [])

CFTC_REPORT_FAMILIES = {
    "legacy_futures_only": {
        "enabled": True,
        "label": "Futures Only Reports - Legacy COT",
        "start_year": 2000,
        "end_year": CURRENT_YEAR,
        "patterns": {
            "xls": [r"/dea_fut_xls_(\d{4})\.zip$", r"/deafut_xls_(\d{4})\.zip$"],
            "txt": [r"/deacot(\d{4})\.zip$"],
        },
        "bundles": {},
    },
    "legacy_futures_and_options": {
        "enabled": DOWNLOAD_LEGACY_FUTURES_AND_OPTIONS,
        "label": "Futures-and-Options Combined - Legacy COT",
        "start_year": 2000,
        "end_year": CURRENT_YEAR,
        "patterns": {
            "xls": [r"/dea_com_xls_(\d{4})\.zip$", r"/deacom_xls_(\d{4})\.zip$"],
            "txt": [r"/deahistfo_?(\d{4})\.zip$"],
        },
        "bundles": {},
    },
    "disaggregated_futures_only": {
        "enabled": True,
        "label": "Disaggregated Futures Only",
        "start_year": 2017,
        "end_year": CURRENT_YEAR,
        "patterns": {
            "xls": [r"/fut_disagg_xls_(\d{4})\.zip$"],
            "txt": [r"/fut_disagg_txt_(\d{4})\.zip$"],
        },
        # The CFTC page provides this bundle for the early disaggregated history.
        "bundles": {
            "xls": [r"/fut_disagg_xls_hist_2006_2016\.zip$"],
            "txt": [r"/fut_disagg_txt_hist_2006_2016\.zip$"],
        },
    },
    "disaggregated_futures_and_options": {
        "enabled": DOWNLOAD_DISAGGREGATED_FUTURES_AND_OPTIONS,
        "label": "Disaggregated Futures-and-Options Combined",
        "start_year": 2017,
        "end_year": CURRENT_YEAR,
        "patterns": {
            "xls": [r"/com_disagg_xls_(\d{4})\.zip$"],
            "txt": [r"/com_disagg_txt_(\d{4})\.zip$"],
        },
        "bundles": {
            "xls": [r"/com_disagg_xls_hist_2006_2016\.zip$"],
            "txt": [r"/com_disagg_txt_hist_2006_2016\.zip$"],
        },
    },
    "commodity_index_trader_supplement": {
        "enabled": DOWNLOAD_COMMODITY_INDEX_TRADER_SUPPLEMENT,
        "label": "Commodity Index Trader Supplement",
        "start_year": 2006,
        "end_year": CURRENT_YEAR,
        "patterns": {
            "xls": [r"/dea_cit_xls_(\d{4})\.zip$"],
            "txt": [r"/dea_cit_txt_(\d{4})\.zip$"],
        },
        "bundles": {},
    },
}


def _matching_year(path, patterns):
    for pattern in patterns:
        match = re.search(pattern, path)
        if match:
            return int(match.group(1))
    return None


def _matches_any(path, patterns):
    return any(re.search(pattern, path) for pattern in patterns)


def build_cftc_download_manifest(
    page_url=CFTC_HISTORICAL_COMPRESSED_URL,
    report_families=CFTC_REPORT_FAMILIES,
    file_formats=CFTC_FILE_FORMATS,
):
    response = requests.get(
        page_url,
        timeout=30,
        headers={"User-Agent": "Mozilla/5.0 ArabicaFutures research downloader"},
    )
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    hrefs = [a.get("href") for a in soup.find_all("a", href=True)]
    hrefs = [href for href in hrefs if href and href.endswith(".zip")]

    rows = []
    for dataset_key, spec in report_families.items():
        if not spec["enabled"]:
            continue

        for file_format in file_formats:
            for href in hrefs:
                url = urljoin(page_url, href)
                url_path = urlparse(url).path
                filename = Path(url_path).name

                if _matches_any(url_path, spec.get("bundles", {}).get(file_format, [])):
                    rows.append(
                        {
                            "dataset": dataset_key,
                            "label": spec["label"],
                            "file_format": file_format,
                            "year": None,
                            "period": "history_bundle",
                            "url": url,
                            "filename": filename,
                            "destination": str(CFTC_DATA_DIR / dataset_key / filename),
                        }
                    )
                    continue

                year = _matching_year(url_path, spec.get("patterns", {}).get(file_format, []))
                if year is None:
                    continue
                if spec["start_year"] <= year <= spec["end_year"]:
                    rows.append(
                        {
                            "dataset": dataset_key,
                            "label": spec["label"],
                            "file_format": file_format,
                            "year": year,
                            "period": str(year),
                            "url": url,
                            "filename": filename,
                            "destination": str(CFTC_DATA_DIR / dataset_key / filename),
                        }
                    )

    manifest = pd.DataFrame(rows).drop_duplicates(subset=["url", "destination"])
    if manifest.empty:
        raise RuntimeError("No CFTC ZIP links matched the enabled report-family configuration.")

    return manifest.sort_values(["dataset", "file_format", "year", "filename"], na_position="first").reset_index(drop=True)


cftc_manifest = build_cftc_download_manifest()
print(f"Found {len(cftc_manifest)} CFTC ZIP files to download into {CFTC_DATA_DIR}/")
print(cftc_manifest.groupby(["dataset", "file_format"]).size().rename("files").to_string())
cftc_manifest.head()


Found 38 CFTC ZIP files to download into data/cftc_cot/
dataset                     file_format
disaggregated_futures_only  xls            11
legacy_futures_only         xls            27


,dataset,label,file_format,year,period,url,filename,destination
0,disaggregated_futures_only,Disaggregated Futures Only,xls,NaN,history_bundle,https://www.cftc.gov/files/dea/history/fut_dis...,fut_disagg_xls_hist_2006_2016.zip,data/cftc_cot/disaggregated_futures_only/fut_d...
1,disaggregated_futures_only,Disaggregated Futures Only,xls,2017.0,2017,https://www.cftc.gov/files/dea/history/fut_dis...,fut_disagg_xls_2017.zip,data/cftc_cot/disaggregated_futures_only/fut_d...
2,disaggregated_futures_only,Disaggregated Futures Only,xls,2018.0,2018,https://www.cftc.gov/files/dea/history/fut_dis...,fut_disagg_xls_2018.zip,data/cftc_cot/disaggregated_futures_only/fut_d...
3,disaggregated_futures_only,Disaggregated Futures Only,xls,2019.0,2019,https://www.cftc.gov/files/dea/history/fut_dis...,fut_disagg_xls_2019.zip,data/cftc_cot/disaggregated_futures_only/fut_d...
4,disaggregated_futures_only,Disaggregated Futures Only,xls,2020.0,2020,https://www.cftc.gov/files/dea/history/fut_dis...,fut_disagg_xls_2020.zip,data/cftc_cot/disaggregated_futures_only/fut_d...


In [7]:
def download_cftc_zip_manifest(manifest, overwrite=False, pause_seconds=0.2):
    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0 ArabicaFutures research downloader"})

    results = []
    for row in manifest.itertuples(index=False):
        destination = Path(row.destination)
        destination.parent.mkdir(parents=True, exist_ok=True)

        if destination.exists() and destination.stat().st_size > 0 and not overwrite:
            status = "skipped_existing"
        else:
            tmp_destination = destination.with_suffix(destination.suffix + ".part")
            with session.get(row.url, stream=True, timeout=90) as response:
                response.raise_for_status()
                with tmp_destination.open("wb") as file:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            file.write(chunk)
            tmp_destination.replace(destination)
            status = "downloaded"
            time.sleep(pause_seconds)

        results.append(
            {
                "dataset": row.dataset,
                "file_format": row.file_format,
                "period": row.period,
                "status": status,
                "path": str(destination),
                "size_mb": round(destination.stat().st_size / (1024 * 1024), 2),
            }
        )

    return pd.DataFrame(results)


cftc_downloads = download_cftc_zip_manifest(cftc_manifest, overwrite=False)
print("CFTC download summary:")
print(cftc_downloads.groupby(["dataset", "status"]).size().rename("files").to_string())
cftc_downloads.head()


CFTC download summary:
dataset                     status    
disaggregated_futures_only  downloaded    11
legacy_futures_only         downloaded    27


,dataset,file_format,period,status,path,size_mb
0,disaggregated_futures_only,xls,history_bundle,downloaded,data/cftc_cot/disaggregated_futures_only/fut_d...,20.91
1,disaggregated_futures_only,xls,2017,downloaded,data/cftc_cot/disaggregated_futures_only/fut_d...,6.51
2,disaggregated_futures_only,xls,2018,downloaded,data/cftc_cot/disaggregated_futures_only/fut_d...,6.98
3,disaggregated_futures_only,xls,2019,downloaded,data/cftc_cot/disaggregated_futures_only/fut_d...,7.03
4,disaggregated_futures_only,xls,2020,downloaded,data/cftc_cot/disaggregated_futures_only/fut_d...,6.89


## Extract downloaded CFTC ZIP files

Extracts every CFTC ZIP downloaded under `data/cftc_cot/` into `data/raw_cot_data/`. Files are grouped by dataset and ZIP archive name to prevent filename collisions across report families and years.


In [8]:
from pathlib import Path
import zipfile

RAW_COT_DATA_DIR = Path("data/raw_cot_data")


def extract_cftc_zip_files(
    zip_root=CFTC_DATA_DIR,
    output_root=RAW_COT_DATA_DIR,
    overwrite=False,
):
    zip_root = Path(zip_root)
    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)

    zip_paths = sorted(zip_root.rglob("*.zip"))
    if not zip_paths:
        raise FileNotFoundError(
            f"No ZIP files found under {zip_root}. Run the CFTC download cell first."
        )

    results = []
    for zip_path in zip_paths:
        dataset_name = zip_path.parent.name
        archive_name = zip_path.stem
        extract_dir = output_root / dataset_name / archive_name
        extract_dir.mkdir(parents=True, exist_ok=True)

        extracted_files = []
        skipped_files = []
        with zipfile.ZipFile(zip_path) as archive:
            for member in archive.infolist():
                if member.is_dir():
                    continue

                member_name = Path(member.filename).name
                if not member_name:
                    continue

                target_path = extract_dir / member_name
                if target_path.exists() and not overwrite:
                    skipped_files.append(str(target_path))
                    continue

                target_path.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as source, target_path.open("wb") as target:
                    target.write(source.read())
                extracted_files.append(str(target_path))

        results.append(
            {
                "zip_file": str(zip_path),
                "dataset": dataset_name,
                "extract_dir": str(extract_dir),
                "extracted_files": len(extracted_files),
                "skipped_existing_files": len(skipped_files),
            }
        )

    return pd.DataFrame(results)


cot_extraction_results = extract_cftc_zip_files(overwrite=False)
print(f"Extracted CFTC ZIP contents into {RAW_COT_DATA_DIR}/")
print(cot_extraction_results.groupby("dataset")[["extracted_files", "skipped_existing_files"]].sum().to_string())
cot_extraction_results.head()


Extracted CFTC ZIP contents into data/raw_cot_data/
                            extracted_files  skipped_existing_files
dataset                                                            
disaggregated_futures_only               12                       0
legacy_futures_only                      27                       0


,zip_file,dataset,extract_dir,extracted_files,skipped_existing_files
0,data/cftc_cot/disaggregated_futures_only/fut_d...,disaggregated_futures_only,data/raw_cot_data/disaggregated_futures_only/fut_di...,1,0
1,data/cftc_cot/disaggregated_futures_only/fut_d...,disaggregated_futures_only,data/raw_cot_data/disaggregated_futures_only/fut_di...,1,0
2,data/cftc_cot/disaggregated_futures_only/fut_d...,disaggregated_futures_only,data/raw_cot_data/disaggregated_futures_only/fut_di...,1,0
3,data/cftc_cot/disaggregated_futures_only/fut_d...,disaggregated_futures_only,data/raw_cot_data/disaggregated_futures_only/fut_di...,1,0
4,data/cftc_cot/disaggregated_futures_only/fut_d...,disaggregated_futures_only,data/raw_cot_data/disaggregated_futures_only/fut_di...,1,0


## Filter COFFEE C COT rows

Scans every table extracted under `data/raw_cot_data/`, keeps rows containing `COFFEE C`, and writes the filtered outputs to `data/COT/`.


In [11]:
from pathlib import Path
import re

COT_OUTPUT_DIR = Path("data/COT")
COFFEE_C_FILTER_TEXT = "COFFEE C"


def _safe_filename(value):
    value = str(value).strip().replace(" ", "_")
    value = re.sub(r"[^A-Za-z0-9_.-]+", "_", value)
    return value.strip("_") or "cot_file"


def read_cot_table(path):
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix in {".xls", ".xlsx"}:
        return pd.read_excel(path)

    if suffix in {".csv", ".txt"}:
        for kwargs in (
            {"sep": None, "engine": "python"},
            {"sep": ","},
            {"sep": "\t"},
        ):
            try:
                return pd.read_csv(path, **kwargs)
            except Exception:
                continue

    raise ValueError(f"Unsupported or unreadable COT file format: {path}")


def filter_coffee_c_cot_data(
    raw_root=RAW_COT_DATA_DIR,
    output_root=COT_OUTPUT_DIR,
    filter_text=COFFEE_C_FILTER_TEXT,
):
    raw_root = Path(raw_root)
    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)

    table_paths = sorted(
        path
        for path in raw_root.rglob("*")
        if path.is_file() and path.suffix.lower() in {".xls", ".xlsx", ".csv", ".txt"}
    )
    if not table_paths:
        raise FileNotFoundError(
            f"No extracted COT table files found under {raw_root}. Run the extraction cell first."
        )

    filtered_frames = []
    manifest_rows = []

    for table_path in table_paths:
        relative_path = table_path.relative_to(raw_root)
        dataset = relative_path.parts[0] if len(relative_path.parts) > 1 else "unknown_dataset"
        archive_name = relative_path.parts[1] if len(relative_path.parts) > 2 else table_path.stem

        try:
            df = read_cot_table(table_path)
        except Exception as exc:
            manifest_rows.append(
                {
                    "dataset": dataset,
                    "source_file": str(table_path),
                    "output_file": None,
                    "source_rows": None,
                    "coffee_c_rows": 0,
                    "status": "read_error",
                    "error": str(exc),
                }
            )
            continue

        row_text = df.apply(
            lambda row: " ".join(
                "" if pd.isna(value) else str(value)
                for value in row
            ),
            axis=1,
        )
        mask = row_text.str.contains(re.escape(filter_text), case=False, na=False)
        coffee_df = df.loc[mask].copy()

        if coffee_df.empty:
            output_file = None
            status = "no_matching_rows"
        else:
            coffee_df.insert(0, "source_file", str(table_path))
            coffee_df.insert(1, "source_dataset", dataset)
            coffee_df.insert(2, "source_archive", archive_name)

            dataset_output_dir = output_root / dataset
            dataset_output_dir.mkdir(parents=True, exist_ok=True)
            output_file = dataset_output_dir / f"{_safe_filename(archive_name)}__{_safe_filename(table_path.stem)}__coffee_c.csv"
            coffee_df.to_csv(output_file, index=False)
            filtered_frames.append(coffee_df)
            status = "filtered"

        manifest_rows.append(
            {
                "dataset": dataset,
                "source_file": str(table_path),
                "output_file": str(output_file) if output_file else None,
                "source_rows": len(df),
                "coffee_c_rows": len(coffee_df),
                "status": status,
                "error": None,
            }
        )

    manifest = pd.DataFrame(manifest_rows)
    manifest.to_csv(output_root / "coffee_c_filter_manifest.csv", index=False)

    if filtered_frames:
        combined = pd.concat(filtered_frames, ignore_index=True, sort=False)
        combined.to_csv(output_root / "coffee_c_all_cot_data.csv", index=False)

        for dataset, dataset_df in combined.groupby("source_dataset", dropna=False):
            dataset_file = output_root / f"{_safe_filename(dataset)}__coffee_c_all.csv"
            dataset_df.to_csv(dataset_file, index=False)
    else:
        combined = pd.DataFrame()

    return manifest, combined


coffee_c_manifest, coffee_c_all = filter_coffee_c_cot_data()
print(f"Wrote COFFEE C filtered COT outputs into {COT_OUTPUT_DIR}/")
print(coffee_c_manifest.groupby(["dataset", "status"])["coffee_c_rows"].agg(["count", "sum"]).to_string())
print(f"Total COFFEE C rows: {len(coffee_c_all)}")
coffee_c_manifest.head()


WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
Wrote COFFEE C filtered COT outputs into data/COT/
                                     count   sum
dataset                    status               
disaggregated_futures_only filtered     12  1056
legacy_futures_only        filtered     27  1391
Total COFFEE C rows: 2447


,dataset,source_file,output_file,source_rows,coffee_c_rows,status,error
0,disaggregated_futures_only,data/raw_cot_data/disaggregated_futures_only/fut_di...,data/COT/disaggregated_futures_only/fut_disagg_xls_...,10204,52,filtered,None
1,disaggregated_futures_only,data/raw_cot_data/disaggregated_futures_only/fut_di...,data/COT/disaggregated_futures_only/fut_disagg_xls_...,10874,53,filtered,None
2,disaggregated_futures_only,data/raw_cot_data/disaggregated_futures_only/fut_di...,data/COT/disaggregated_futures_only/fut_disagg_xls_...,10954,52,filtered,None
3,disaggregated_futures_only,data/raw_cot_data/disaggregated_futures_only/fut_di...,data/COT/disaggregated_futures_only/fut_disagg_xls_...,10786,52,filtered,None
4,disaggregated_futures_only,data/raw_cot_data/disaggregated_futures_only/fut_di...,data/COT/disaggregated_futures_only/fut_disagg_xls_...,10704,52,filtered,None


### The resource of events are too big for my laptop to accomodate 

https://data.gdeltproject.org/events/index.html

> Guess that's why people love compute haha 
>
> Src: https://blog.gdeltproject.org/the-datasets-of-gdelt-as-of-february-2016/